In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jeanmidev/smart-meters-in-london")

print("Path to dataset files:", path)


100%|██████████| 1.17G/1.17G [00:11<00:00, 106MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21


In [2]:
import os

dataset_path = "/root/.cache/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21"

print("Daftar file dalam dataset:")
print(os.listdir(dataset_path))

!pip install pyspark

Daftar file dalam dataset:
['daily_dataset.csv', 'acorn_details.csv', 'darksky_parameters_documentation.html', 'daily_dataset', 'uk_bank_holidays.csv', 'weather_daily_darksky.csv', 'halfhourly_dataset', 'hhblock_dataset', 'weather_hourly_darksky.csv', 'informations_households.csv']


In [3]:
from pyspark.sql import SparkSession

# 1. Membuka sesi Spark untuk baca data di Colab
spark = SparkSession.builder \
    .appName("Load Forecasting Smart Grid - Replikasi Parsial") \
    .master("local[*]") \
    .getOrCreate()

print("SparkSession berhasil dibuat!\n")

# Path ke direktori dataset
dataset_path = "/root/.cache/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21/"

# 2. Memuat data konsumsi listrik harian dan data cuaca
df_energy = spark.read.csv(dataset_path + "daily_dataset.csv", header=True, inferSchema=True)
df_weather = spark.read.csv(dataset_path + "weather_daily_darksky.csv", header=True, inferSchema=True)

# 3. Menampilkan skema dan 5 baris awal dari masing-masing data
print("--- Data Konsumsi Listrik (Energy) ---")
df_energy.printSchema()
df_energy.show(5)

print("\n--- Data Cuaca (Weather) ---")
df_weather.printSchema()
df_weather.show(5)

SparkSession berhasil dibuat!

--- Data Konsumsi Listrik (Energy) ---
root
 |-- LCLid: string (nullable = true)
 |-- day: date (nullable = true)
 |-- energy_median: double (nullable = true)
 |-- energy_mean: double (nullable = true)
 |-- energy_max: double (nullable = true)
 |-- energy_count: integer (nullable = true)
 |-- energy_std: double (nullable = true)
 |-- energy_sum: double (nullable = true)
 |-- energy_min: double (nullable = true)

+---------+----------+-------------+-------------------+------------------+------------+-------------------+------------------+-------------------+
|    LCLid|       day|energy_median|        energy_mean|        energy_max|energy_count|         energy_std|        energy_sum|         energy_min|
+---------+----------+-------------+-------------------+------------------+------------+-------------------+------------------+-------------------+
|MAC000131|2011-12-15|        0.485|0.43204545454545457|             0.868|          22|0.23914579678767536| 

In [4]:
from pyspark.sql.functions import col, to_date

# Ubah format kolom 'time' di data cuaca menjadi Date agar cocok dengan kolom 'day' di data energi
df_weather_clean = df_weather.withColumn("date_key", to_date(col("time")))

# Ubah nama kolom 'day' di data energi agar seragam
df_energy_clean = df_energy.withColumnRenamed("day", "date_key")

# Gabungkan kedua data (Inner Join) berdasarkan tanggal (date_key)
df_joined = df_energy_clean.join(df_weather_clean, on="date_key", how="inner")

# Pilih kolom yang relevan untuk diprediksi (misalnya: rata-rata energi, suhu maksimal, tutupan awan)
# Sesuai artikel referensi, cuaca sangat memengaruhi konsumsi
df_final = df_joined.select(
    "date_key",
    "LCLid",
    "energy_mean",
    "temperatureMax",
    "cloudCover",
    "humidity"
)

# Buang baris yang memiliki nilai kosong (NaN/Null) agar model ML tidak eror
df_final = df_final.dropna()

print("Data berhasil digabung dan dibersihkan!")
df_final.show(5)
print(f"Total baris data siap pakai: {df_final.count()}")

Data berhasil digabung dan dibersihkan!
+----------+---------+-------------------+--------------+----------+--------+
|  date_key|    LCLid|        energy_mean|temperatureMax|cloudCover|humidity|
+----------+---------+-------------------+--------------+----------+--------+
|2011-12-15|MAC000131|0.43204545454545457|          7.97|      0.42|    0.77|
|2011-12-16|MAC000131|0.29616666875000003|          4.68|       0.7|    0.88|
|2011-12-17|MAC000131|          0.1898125|          5.35|      0.37|    0.86|
|2011-12-18|MAC000131| 0.2189791666666666|          5.49|      0.22|    0.84|
|2011-12-19|MAC000131|0.32597916666666665|          6.64|      0.47|    0.94|
+----------+---------+-------------------+--------------+----------+--------+
only showing top 5 rows
Total baris data siap pakai: 3501189


In [5]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Menggabungkan kolom fitur cuaca menjadi satu kolom vektor ('features')
feature_columns = ["temperatureMax", "cloudCover", "humidity"]
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

# Menerapkan transformasi pada dataframe
df_ml = assembler.transform(df_final)

# 2. Membagi data menjadi Data Latih (80%) dan Data Uji (20%)
# Menggunakan seed=42 agar pembagian acaknya tetap konsisten jika diulang
train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)
print(f"Jumlah Data Latih: {train_data.count()}")
print(f"Jumlah Data Uji: {test_data.count()}\n")

# 3. Membangun dan melatih model Decision Tree
print("Sedang melatih model Decision Tree... (Ini mungkin memakan waktu beberapa saat)")
dt_algorithm = DecisionTreeRegressor(featuresCol="features", labelCol="energy_mean", seed=42)
dt_model = dt_algorithm.fit(train_data)

# 4. Melakukan prediksi pada Data Uji
predictions = dt_model.transform(test_data)

# Menampilkan hasil prediksi (kolom label asli vs hasil tebakan model)
print("--- Hasil Prediksi Konsumsi Listrik ---")
predictions.select("features", "energy_mean", "prediction").show(5)

# 5. Evaluasi performa model menggunakan metrik RMSE (Root Mean Squared Error)
evaluator = RegressionEvaluator(labelCol="energy_mean", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)

print(f"Akurasi Model (RMSE): {rmse}")
print("Semakin rendah nilai RMSE, semakin akurat tebakan modelnya!")

Jumlah Data Latih: 2800917
Jumlah Data Uji: 700272

Sedang melatih model Decision Tree... (Ini mungkin memakan waktu beberapa saat)
--- Hasil Prediksi Konsumsi Listrik ---
+-----------------+-------------------+-------------------+
|         features|        energy_mean|         prediction|
+-----------------+-------------------+-------------------+
|[10.36,0.36,0.93]|0.29650000000000004|0.23829183557349032|
|[12.93,0.41,0.89]| 0.5051785714285714|0.22928651251574414|
|[12.93,0.41,0.89]|            0.32325|0.22928651251574414|
|[13.03,0.48,0.79]| 0.6264791666666667|0.22928651251574414|
|[13.03,0.48,0.79]|0.06725000000000002|0.22928651251574414|
+-----------------+-------------------+-------------------+
only showing top 5 rows
Akurasi Model (RMSE): 0.1875542414609251
Semakin rendah nilai RMSE, semakin akurat tebakan modelnya!
